# MedCLIP-SAMv2 Thigh Segmentation — Lambda

Runs [MedCLIP-SAMv2](https://github.com/HealthX-Lab/MedCLIP-SAMv2) zero-shot segmentation
on **fat-fraction** and **water** Dixon MRI stacks.

Pipeline per stack per muscle:
1. Export NIfTI slices to PNG
2. BiomedCLIP saliency map (text prompt → heatmap per slice)
3. Postprocessing (kmeans → coarse binary mask)
4. SAM refinement (coarse mask → precise boundary)
5. Reassemble PNG masks → 3D NPZ

**Note:** BiomedCLIP does not reliably distinguish left vs right from
image content alone. The L/R prompts are best-effort; evaluate results
carefully.

## Before running — upload to Lambda

```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  ubuntu@129.80.59.179:~/
```

## Download results when done

```bash
# fat fraction
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@129.80.59.179:~/medclipsamv2_segs_fatfrac/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2_segs_fatfrac/

# water
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@129.80.59.179:~/medclipsamv2_segs_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2_segs_water/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os

REPO_DIR  = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR  = os.path.expanduser('~/mcsam2_env')
VENV_PY   = os.path.join(VENV_DIR, 'bin', 'python')

# ── Clone repo ───────────────────────────────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])
    print('Cloned MedCLIP-SAMv2')
else:
    print('Repo already present')

# ── Create a clean venv (no --system-site-packages) ─────────────────────────
if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])
    print('Venv created:', VENV_DIR)
else:
    print('Venv already exists:', VENV_DIR)

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')

# numpy 2.x + sklearn — fresh builds, no binary incompatibility
venv_pip('install', '-q', 'numpy', 'scikit-learn')

# PyTorch with CUDA
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')

# SAM (bundled in repo)
sam_dir = os.path.join(REPO_DIR, 'segment-anything')
venv_pip('install', '-q', '-e', sam_dir)

# pydensecrf — compiled C extension, must be built from source (no PyPI wheel)
venv_pip('install', '-q',
    'git+https://github.com/lucasb-eyer/pydensecrf.git')

# All packages imported by the MedCLIP-SAMv2 scripts.
# transformers pinned <4.46 — newer versions changed CLIPConfig to keyword-only
# args, breaking BiomedCLIP's cached configuration_biomed_clip.py.
venv_pip('install', '-q',
    'open_clip_torch',
    'opencv-python',
    'SimpleITK',
    'Pillow',
    'huggingface_hub',
    'transformers<4.46',
    'matplotlib',
    'grad-cam',
    'pandas',
    'tqdm',
    'scipy',
)

print('All dependencies installed.')

# ── Verify — override Jupyter's MPLBACKEND so matplotlib imports headlessly ──
_verify_env = {k: v for k, v in os.environ.items() if k != 'MPLBACKEND'}
_verify_env['MPLBACKEND'] = 'Agg'
_verify_env['PYTHONNOUSERSITE'] = '1'

result = subprocess.run([VENV_PY, '-c',
    'import numpy, sklearn, torch, cv2, matplotlib, pytorch_grad_cam, pandas, pydensecrf; '
    'print("numpy", numpy.__version__); '
    'print("sklearn", sklearn.__version__); '
    'print("torch", torch.__version__, "cuda", torch.cuda.is_available()); '
    'print("cv2", cv2.__version__); '
    'print("pandas", pandas.__version__); '
    'print("pytorch_grad_cam OK"); '
    'print("pydensecrf OK")'],
    capture_output=True, text=True, env=_verify_env)
print(result.stdout)
if result.returncode != 0:
    print('VERIFY STDERR:', result.stderr[-1000:])

In [ ]:
import os, urllib.request

CKPT_DIR  = os.path.join(REPO_DIR, 'segment-anything', 'sam_checkpoints')
CKPT_FILE = os.path.join(CKPT_DIR, 'sam_vit_b_01ec64.pth')
# Using ViT-B for speed; swap URL for ViT-H (sam_vit_h_4b8939.pth) for better quality
URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_FILE):
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    urllib.request.urlretrieve(URL, CKPT_FILE)
    print(f'Done ({os.path.getsize(CKPT_FILE) // 1_000_000} MB)')
else:
    print('SAM checkpoint already present')

In [ ]:
import glob, shutil, tempfile
import numpy as np
import SimpleITK as sitk
import torch
import cv2
from PIL import Image

# --- paths ---
DATA_ROOT  = os.path.expanduser('~/myosegmenTUM')
REPO_DIR   = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY    = os.path.expanduser('~/mcsam2_env/bin/python')
SAM_CKPT   = os.path.join(REPO_DIR, 'segment-anything', 'sam_checkpoints', 'sam_vit_b_01ec64.pth')
SAM_TYPE   = 'vit_b'

OUTPUT_DIRS = {
    'FATFRACTION': os.path.expanduser('~/medclipsamv2_segs_fatfrac'),
    'WATER':       os.path.expanduser('~/medclipsamv2_segs_water'),
}
IMAGE_GLOBS = {
    'FATFRACTION': os.path.join(DATA_ROOT, '*/ImageData/*FATFRACTION/*FATFRACTION_stack*.nii'),
    'WATER':       os.path.join(DATA_ROOT, '*/ImageData/*_WATER/*_WATER_stack*.nii'),
}

for d in OUTPUT_DIRS.values():
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device  :', DEVICE)
print('VENV_PY :', VENV_PY, '— exists:', os.path.exists(VENV_PY))
print('SAM ckpt:', os.path.exists(SAM_CKPT))

In [ ]:
# Muscle definitions: (output_key, text_prompt)
# BiomedCLIP works with descriptive image captions, not instructions.
# Prompts are intentionally descriptive of what the muscle looks like in MRI.
MUSCLES = [
    ('R_gracilis',
     'gracilis muscle right thigh Dixon MRI axial cross section'),
    ('L_gracilis',
     'gracilis muscle left thigh Dixon MRI axial cross section'),
    ('R_sartorius',
     'sartorius muscle right thigh Dixon MRI axial cross section'),
    ('L_sartorius',
     'sartorius muscle left thigh Dixon MRI axial cross section'),
]

# Scripts called in order
SAL_SCRIPT  = os.path.join(REPO_DIR, 'saliency_maps', 'generate_saliency_maps.py')
POST_SCRIPT = os.path.join(REPO_DIR, 'postprocessing', 'postprocess_saliency_maps.py')
SAM_SCRIPT  = os.path.join(REPO_DIR, 'segment-anything', 'prompt_sam.py')

for s in [SAL_SCRIPT, POST_SCRIPT, SAM_SCRIPT]:
    print('exists:', os.path.exists(s), s)

In [ ]:
def export_slices_as_png(img_array, out_dir):
    """Write (D, H, W) float array as 0-indexed RGB PNGs for MedCLIP-SAMv2."""
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        rgb = np.stack([sl_uint8] * 3, axis=-1)
        Image.fromarray(rgb).save(os.path.join(out_dir, f'{i}.png'))


# Build subprocess env that puts venv site-packages FIRST.
import glob as _glob
_venv_site = _glob.glob(os.path.join(VENV_DIR, 'lib', 'python3.*', 'site-packages'))
if not _venv_site:
    raise RuntimeError(f'Could not find site-packages inside {VENV_DIR}')
VENV_SITE = _venv_site[0]
print('Venv site-packages:', VENV_SITE)

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = VENV_SITE + ':' + SUBPROCESS_ENV.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
# Jupyter sets MPLBACKEND=module://matplotlib_inline.backend_inline which fails
# headlessly in subprocesses. Override with a non-interactive backend.
SUBPROCESS_ENV['MPLBACKEND'] = 'Agg'


def run_stage(cmd, stdin_text=None, cwd=None):
    """Run a subprocess via the venv python with venv packages guaranteed first."""
    full_cmd = [VENV_PY if c == sys.executable else c for c in cmd]
    result = subprocess.run(
        full_cmd,
        input=stdin_text,
        text=True,
        capture_output=True,
        cwd=cwd or REPO_DIR,
        env=SUBPROCESS_ENV,
    )
    if result.returncode != 0:
        print('STDOUT:', result.stdout[-2000:])
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError(f'Command failed (exit {result.returncode}): {full_cmd}')


def load_mask_pngs(mask_dir, num_slices, H, W):
    """Load numbered PNG masks back into a (D, H, W) uint8 volume."""
    vol = np.zeros((num_slices, H, W), dtype=np.uint8)
    for i in range(num_slices):
        png_path = os.path.join(mask_dir, f'{i}.png')
        if os.path.exists(png_path):
            mask_img = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
            if mask_img is not None:
                if mask_img.shape != (H, W):
                    mask_img = cv2.resize(mask_img, (W, H),
                                          interpolation=cv2.INTER_NEAREST)
                vol[i] = (mask_img > 127).astype(np.uint8)
    return vol


print('Helper functions defined.')

In [ ]:
for modality, image_glob in IMAGE_GLOBS.items():
    output_dir = OUTPUT_DIRS[modality]
    image_files = sorted(glob.glob(image_glob))
    print(f'\n═══ {modality}: {len(image_files)} stacks ═══')

    for nii_path in image_files:
        stem     = os.path.splitext(os.path.basename(nii_path))[0]
        out_path = os.path.join(output_dir, f'{stem}_medclipsamv2.npz')

        if os.path.exists(out_path):
            print(f'Skipping (done): {stem}')
            continue

        print(f'\nProcessing: {stem}')
        img_sitk  = sitk.ReadImage(nii_path)
        img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (D, H, W)
        D, H, W   = img_array.shape
        print(f'  Shape: {img_array.shape}')

        tmp_root = tempfile.mkdtemp(prefix='mcsam2_')
        all_masks = {}

        try:
            # export slices once, reuse for all muscles
            png_dir = os.path.join(tmp_root, 'slices')
            export_slices_as_png(img_array, png_dir)

            for muscle_name, text_prompt in MUSCLES:
                print(f'  [{muscle_name}] prompt: "{text_prompt}"')

                sal_dir  = os.path.join(tmp_root, f'sal_{muscle_name}')
                post_dir = os.path.join(tmp_root, f'post_{muscle_name}')
                sam_dir  = os.path.join(tmp_root, f'sam_{muscle_name}')
                os.makedirs(sal_dir,  exist_ok=True)
                os.makedirs(post_dir, exist_ok=True)
                os.makedirs(sam_dir,  exist_ok=True)

                # Stage 1: BiomedCLIP saliency maps
                # generate_saliency_maps.py calls input() to read the text prompt
                run_stage(
                    [sys.executable, SAL_SCRIPT,
                     '--input-path',  png_dir,
                     '--output-path', sal_dir,
                     '--val-path',    png_dir,
                     '--model-name',  'BiomedCLIP',
                     '--device',      DEVICE],
                    stdin_text=text_prompt + '\n',
                )

                # Stage 2: postprocess saliency maps → coarse binary masks
                run_stage(
                    [sys.executable, POST_SCRIPT,
                     '--input-path',  png_dir,
                     '--output-path', post_dir,
                     '--sal-path',    sal_dir,
                     '--postprocess', 'kmeans',
                     '--filter'],
                )

                # Stage 3: SAM refinement
                run_stage(
                    [sys.executable, SAM_SCRIPT,
                     '--input',      png_dir,
                     '--mask-input', post_dir,
                     '--output',     sam_dir,
                     '--model-type', SAM_TYPE,
                     '--checkpoint', SAM_CKPT,
                     '--prompts',    'boxes',
                     '--device',     DEVICE],
                )

                # collect PNG masks → 3D volume
                all_masks[muscle_name] = load_mask_pngs(sam_dir, D, H, W)
                voxels = int(all_masks[muscle_name].sum())
                print(f'    -> {voxels:,} positive voxels')

            np.savez_compressed(out_path, **all_masks)
            print(f'  Saved -> {out_path}')

        except Exception as e:
            print(f'  ERROR on {stem}: {e}')

        finally:
            shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')

In [ ]:
# sanity check
for modality, output_dir in OUTPUT_DIRS.items():
    results = sorted(glob.glob(os.path.join(output_dir, '*.npz')))
    print(f'\n{modality}: {len(results)} output files')
    if results:
        sample = np.load(results[0])
        print(f'  Sample: {results[0]}')
        for name in sorted(sample.files):
            arr = sample[name]
            print(f'    {name}: shape={arr.shape}  voxels={int(arr.sum()):,}')